# P4 - Réaliser une étude de Santé Publique avec Python

## Introduction

**Contexte** :

Ce projet s’appuie sur des données issues de la FAO. Il mobilise quatre jeux de données afin d’analyser la disponibilité alimentaire, la sous-nutrition, l’aide alimentaire et la population à l’échelle mondiale.

**Objectif** :

Réaliser différentes analyses sur la période 2013–2017 à partir des données disponibles afin d'étudier la problématique de la faim dans le monde.

**Méthodologie** :

Le travail est structuré en deux étapes :

**Notebook N1** : analyse exploratoire des fichiers bruts afin de préparer les données à l'analyse. Cette étape comprend le nettoyage des données, la gestion des valeurs manquantes et des doublons, la vérification des types de données, la standardisation des variables textuelles ainsi que la création de nouvelles variables dérivées.

**Notebook N2** : analyse des fichiers transformés afin de répondre à différentes questions relatives à l’accès à l’alimentation, à la sous-nutrition et à la disponibilité alimentaire.

**Dans ce notebook** : 

1. [Importation des données et chargement des données](#1-importation-des-librairies-et-chargement-des-fichiers)

2. [Analyse exploratoire des données](#2-analyse-exploratoire-des-fichiers)

    2.1. [Fichier Aide Alimentaire](#21-fichier-aide-alimentaire)

    2.2. [Fichier Disponibilité Alimentaire](#22-fichier-disponibilité-alimentaire)

    2.3. [Fichier Population](#23-fichier-population)

    2.4. [Fichier Sous-Nutrition](#24-fichier-sous-nutrition)

_____________

## 1. Importation des librairies et chargement des fichiers

Importation des librairies et chargement des jeux de données dans des dataframes. Paramétrage des options d’affichage et de visualisation.

### 1.1 Importation des librairies

In [444]:
import pandas as pd
import numpy as np

In [ ]:
#paramètres d'affichage numérique
pd.set_option('display.float_format', '{:.0f}'.format)

### 1.2 Chargement des fichiers

In [446]:
df_aide_alimentaire_raw = pd.read_csv('../data/DAN-P4-FAO/aide_alimentaire.csv')
df_dispo_alimentaire_raw = pd.read_csv('../data/DAN-P4-FAO/dispo_alimentaire.csv')
df_population_raw = pd.read_csv('../data/DAN-P4-FAO/population.csv')
df_sous_nutrition_raw =  pd.read_csv('../data/DAN-P4-FAO/sous_nutrition.csv')

_____________

## 2. Analyse exploratoire des fichiers

Nettoyage, transformation et préparation des données en vue de l’analyse (gestion des valeurs manquantes, standardisation, conversions d’unités, création de variables).

### 2.1 Fichier aide alimentaire

**Description du dataset**

Ce dataset présente les quantités d’aide alimentaire en kg reçues par 76 pays bénéficiaires entre 2013 et 2016, détaillées par 16 catégories de produits alimentaires.

Chaque ligne représente un pays, une annéee et un produit unique.

Après nettoyage et transformation, le dataframe aide_alimentaire conserve 1475 enregistrements et 4 colonnes.

#### 2.1.1 Compréhension générale du dataset

In [447]:
#dimensions du dataset
print(f'Le dataframe aide_alimentaire contient :\n{df_aide_alimentaire_raw.shape[0]} enregistrements et {df_aide_alimentaire_raw.shape[1]} colonnes')

Le dataframe aide_alimentaire contient :
1475 enregistrements et 4 colonnes


In [448]:
#structure des colonnes et types
df_aide_alimentaire_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1475 entries, 0 to 1474
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Pays bénéficiaire  1475 non-null   object
 1   Année              1475 non-null   int64 
 2   Produit            1475 non-null   object
 3   Valeur             1475 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 46.2+ KB


In [449]:
#aperçu des données (5 premières lignes)
df_aide_alimentaire_raw.head()

,Pays bénéficiaire,Année,Produit,Valeur
0,Afghanistan,2013,Autres non-céréales,682
1,Afghanistan,2014,Autres non-céréales,335
2,Afghanistan,2013,Blé et Farin,39224
3,Afghanistan,2014,Blé et Farin,15160
4,Afghanistan,2013,Céréales,40504


In [450]:
#nom des colonnes
df_aide_alimentaire_raw.columns

Index(['Pays bénéficiaire', 'Année', 'Produit', 'Valeur'], dtype='object')

#### 2.1.2 Compréhension des variables

In [451]:
#nombre de pays
df_aide_alimentaire_raw['Pays bénéficiaire'].nunique()

76

In [452]:
#années disponibles
df_aide_alimentaire_raw['Année'].unique()

array([2013, 2014, 2015, 2016])

In [453]:
#nombre produits disponibles
df_aide_alimentaire_raw['Produit'].nunique()

16

In [454]:
#produits dispobibles
df_aide_alimentaire_raw['Produit'].unique()

array(['Autres non-céréales', 'Blé et Farin', 'Céréales',
       'Fruits secs, total', 'Huiles végétales',
       'Légumineuses Sèches,Tot.', 'Non-céréales', 'Riz, total',
       'Sucre, total', 'Céréales Secondaires', 'Lait, total',
       'Mélanges et préparations', 'Poiss&produi', 'BulgurFarEnt',
       'Viande Total', 'Graisses Com'], dtype=object)

#### 2.1.3 Audit de qualité

In [455]:
#valeurs manquantes
df_aide_alimentaire_raw.isna().sum()

Pays bénéficiaire    0
Année                0
Produit              0
Valeur               0
dtype: int64

In [456]:
#vérification doublons métier
df_aide_alimentaire_raw[['Pays bénéficiaire','Année','Produit']].duplicated().sum()

np.int64(0)

Conclusion :
* Chaque ligne représente un pays, une année et un produit unique
* Pas de valeurs manquantes
* types adéquats (vérifier variable 'valeur', si cohérent)

#### 2.1.4 Nettoyage et standardisation 

In [457]:
#renommer des colonnes
df_aide_alimentaire_raw = (
    df_aide_alimentaire_raw
    .rename(
        columns={
            'Pays bénéficiaire':'Zone'
            }
    )
)

In [458]:
#nettoyer et standardiser variable Zone
df_aide_alimentaire_raw['Zone'] = (
    df_aide_alimentaire_raw['Zone']
    .str
    .strip()
    .str
    .title()
)

In [459]:
df_aide_alimentaire_raw['Zone'].unique()

array(['Afghanistan', 'Algérie', 'Angola', 'Bangladesh', 'Bénin',
       'Bhoutan', 'Bolivie (État Plurinational De)', 'Burkina Faso',
       'Burundi', 'Cambodge', 'Cameroun', 'Chine, Continentale',
       'Colombie', 'Comores', 'Congo', "Côte D'Ivoire", 'Cuba',
       'Djibouti', 'Égypte', 'El Salvador', 'Équateur', 'Éthiopie',
       'Gambie', 'Géorgie', 'Ghana', 'Guatemala', 'Guinée',
       'Guinée-Bissau', 'Haïti', 'Honduras',
       "Iran (République Islamique D')", 'Iraq', 'Jordanie', 'Kenya',
       'Kirghizistan', 'Lesotho', 'Liban', 'Libéria', 'Libye',
       'Madagascar', 'Malawi', 'Mali', 'Mauritanie', 'Mozambique',
       'Myanmar', 'Népal', 'Nicaragua', 'Niger', 'Ouganda', 'Pakistan',
       'Palestine', 'Philippines', 'République Arabe Syrienne',
       'République Centrafricaine', 'République Démocratique Du Congo',
       'République Démocratique Populaire Lao', 'République Dominicaine',
       'République Populaire Démocratique De Corée',
       'République-Unie De T

In [460]:
#nettoyer et standardiser des catégories produits
df_aide_alimentaire_raw['Produit'] = (
    df_aide_alimentaire_raw['Produit']
    .str
    .strip()
    .str
    .capitalize()
)

In [461]:
df_aide_alimentaire_raw['Produit'].unique().tolist()

['Autres non-céréales',
 'Blé et farin',
 'Céréales',
 'Fruits secs, total',
 'Huiles végétales',
 'Légumineuses sèches,tot.',
 'Non-céréales',
 'Riz, total',
 'Sucre, total',
 'Céréales secondaires',
 'Lait, total',
 'Mélanges et préparations',
 'Poiss&produi',
 'Bulgurfarent',
 'Viande total',
 'Graisses com']

In [462]:
correction = {
    'Légumineuses sèches,tot.':'Légumineuses sèches',
    'Riz, total':'Riz',
    'Sucre, total':'Sucre',
    'Blé et farin':'Blé et farines',
    'Fruits secs, total':'Fruits secs',
    'Bulgurfarent':'Bulgur',
    'Lait, total':'Lait',
    'Poiss&produi':'Poissons et produits de la mer',
    'Viande total':'Viande',
    'Graisses com':'Graisses comestibles'
}
df_aide_alimentaire_raw['Produit'] = (
    df_aide_alimentaire_raw['Produit']
    .replace(correction)
)

In [463]:
df_aide_alimentaire_raw['Produit'].value_counts()

Produit
Non-céréales                      220
Céréales                          200
Huiles végétales                  179
Légumineuses sèches               171
Riz                               147
Mélanges et préparations          142
Autres non-céréales               120
Sucre                              67
Céréales secondaires               61
Blé et farines                     58
Fruits secs                        39
Bulgur                             24
Lait                               23
Poissons et produits de la mer     21
Viande                              2
Graisses comestibles                1
Name: count, dtype: int64

In [464]:
#conversion tonnes/kg
df_aide_alimentaire_raw['Valeur(kg)'] = (
    df_aide_alimentaire_raw['Valeur']*1000
)

In [465]:
#supprimer ancienne colonne
df_aide_alimentaire_raw.drop(
    columns='Valeur', 
    inplace=True
    )

#### 2.1.5 Validation des transformations

In [466]:
#aperçu après nettoyage
df_aide_alimentaire_raw.head()

,Zone,Année,Produit,Valeur(kg)
0,Afghanistan,2013,Autres non-céréales,682000
1,Afghanistan,2014,Autres non-céréales,335000
2,Afghanistan,2013,Blé et farines,39224000
3,Afghanistan,2014,Blé et farines,15160000
4,Afghanistan,2013,Céréales,40504000


In [467]:
# vérification catégories
df_aide_alimentaire_raw[['Zone','Produit']].nunique()

Zone       76
Produit    16
dtype: int64

In [468]:
df_aide_alimentaire_raw.to_csv('../data/data_cleaned/aide_alimentaire_cleaned.csv', index=False)
df_aide_alimentaire = pd.read_csv('../data/data_cleaned/aide_alimentaire_cleaned.csv')

In [469]:
#dimensions du dataframe après transformations

print(f'Après les transformations, le dataframe aide_alimentaire présente :\n{df_aide_alimentaire.shape[0]} enregistrements et {df_aide_alimentaire.shape[1]} colonnes')

Après les transformations, le dataframe aide_alimentaire présente :
1475 enregistrements et 4 colonnes


#### 2.1.6 Transformations réalisées

* Standardisation des variables texte (suppression des espaces inutiles, uniformisation de l'écriture et correction des erreurs orthographiques).
* Renommage de la variable Pays bénéficiaire en Zone.
* Conversion des unités de tonnes en kilogrammes (× 1 000).

#### 2.1.7 Première exploration analytique

Combien de kg d'aide alimentaire chaque pays a-t-il reçu par année, entre 2013 et 2016, et combien de produits différents ont été distribués ?

In [470]:
aide_alimentaire_par_pays = (
    df_aide_alimentaire.groupby(['Zone','Année'])
    .agg(
        nb_produits = ('Produit','nunique'), 
        total_kg = ('Valeur(kg)','sum')
        )
        .reset_index()
)
aide_alimentaire_par_pays.head()

,Zone,Année,nb_produits,total_kg
0,Afghanistan,2013,8,128238000
1,Afghanistan,2014,8,57214000
2,Algérie,2013,11,35234000
3,Algérie,2014,10,18980000
4,Algérie,2015,10,17424000


In [531]:
aide_alimentaire_par_pays.to_csv('../data/tables/aide_alimentaire_par_pays.csv', index=False)

Quelle est la répartition des quantités par produit pour chaque pays et année ?

In [532]:
# répartition des quantités par produit
table_aide_alimentaire_produit = (
    df_aide_alimentaire
    .pivot_table(
        index=['Zone','Année'],
        columns='Produit',
        values='Valeur(kg)',
        aggfunc='sum',
        fill_value=0
        )
)
table_aide_alimentaire_produit.head()

Produit            Autres non-céréales  Blé et farines  Bulgur  Céréales  \
Zone        Année                                                          
Afghanistan 2013                682000        39224000       0  40504000   
            2014                335000        15160000       0  15989000   
Algérie     2013                252000               0       0  10030000   
            2014                205000               0       0   4933000   
            2015                  4000               0       0   3512000   

Produit            Céréales secondaires  Fruits secs  Graisses comestibles  \
Zone        Année                                                            
Afghanistan 2013                      0        85000                     0   
            2014                      0            0                     0   
Algérie     2013                2862000       204000                     0   
            2014                2280000            0                     0   
            2015                1311000       252000                     0   

Produit            Huiles végétales    Lait  Légumineuses sèches  \
Zone        Année                                                  
Afghanistan 2013           11087000       0             11761000   
            2014            8185000       0              4010000   
Algérie     2013            1030000  350000              4111000   
            2014            1050000  582000              2240000   
            2015                  0  500000              3026000   

Produit            Mélanges et préparations  Non-céréales  \
Zone        Année                                           
Afghanistan 2013                          0      23615000   
            2014                          0      12618000   
Algérie     2013                    3112000       7587000   
            2014                    1731000       4557000   
            2015                     559000       5200000   

Produit            Poissons et produits de la mer      Riz    Sucre  Viande  
Zone        Année                                                            
Afghanistan 2013                                0  1280000        0       0  
            2014                                0   829000    88000       0  
Algérie     2013                                0  4056000  1640000       0  
            2014                                0   922000   480000       0  
            2015                                0  1642000  1418000       0

In [533]:
table_aide_alimentaire_produit.to_csv('../data/tables/table_aide_alimentaier_produit.csv', index=False)

_________

### 2.2 Fichier disponibilité alimentaire

**Description du dataset**

Ce dataset présente la disponibilité alimentaire en 2017 pour 174 pays et 98 produits d'origine végétale et animale. Les quantités sont exprimées en kg.

Il correspond à la disponibilité d'aliments en
quantité suffisante et d'une qualité appropriée pour l’alimentation humaine.

La disponibilité intérieure correspond à la disponibilité totale d’un aliment au sein du pays. Elle est liée à l’ensemble des variables décrivant ses différentes utilisations, selon une relation d’équilibre qui peut s’exprimer par l’équation suivante : 

``` Production + Importations - Exportations + Variation de stock = Disponibilité intérieure = Semences + Pertes + Nourriture + Aliments pour animaux + Traitement + Autres utilisations ```

Chaque ligne de ce dataset correspond à un pays et un produit unique.

Après nettoyage et transformations, le dataframe dispo_alimentaire comporte 15605 enregistrements et 19 colonnes.

#### 2.2.1 Compréhension générale du dataset

In [472]:
#dimensions du dataset
nb_lignes = df_dispo_alimentaire_raw.shape[0]
nb_cols = df_dispo_alimentaire_raw.shape[1]

print(f'Le dataframe dispo_alimentaire comporte :\n{nb_lignes} enregistrements et {nb_cols} colonnes')

Le dataframe dispo_alimentaire comporte :
15605 enregistrements et 18 colonnes


In [473]:
#structure des colonnes et types
df_dispo_alimentaire_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15605 entries, 0 to 15604
Data columns (total 18 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   Zone                                                           15605 non-null  object 
 1   Produit                                                        15605 non-null  object 
 2   Origine                                                        15605 non-null  object 
 3   Aliments pour animaux                                          2720 non-null   float64
 4   Autres Utilisations                                            5496 non-null   float64
 5   Disponibilité alimentaire (Kcal/personne/jour)                 14241 non-null  float64
 6   Disponibilité alimentaire en quantité (kg/personne/an)         14015 non-null  float64
 7   Disponibilité de matière grasse en quantité (g/personne/jo

In [474]:
#aperçu des données
df_dispo_alimentaire_raw.head()

,Zone,Produit,Origine,Aliments pour animaux,Autres Utilisations,Disponibilité alimentaire (Kcal/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Pertes,Production,Semences,Traitement,Variation de stock
0,Afghanistan,Abats Comestible,animale,NaN,NaN,5,2,0,1,53,NaN,NaN,53,NaN,53,NaN,NaN,NaN
1,Afghanistan,"Agrumes, Autres",vegetale,NaN,NaN,1,1,0,0,41,2,40,39,2,3,NaN,NaN,NaN
2,Afghanistan,Aliments pour enfants,vegetale,NaN,NaN,1,0,0,0,2,NaN,2,2,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,Ananas,vegetale,NaN,NaN,0,0,NaN,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,Bananes,vegetale,NaN,NaN,4,3,0,0,82,NaN,82,82,NaN,NaN,NaN,NaN,NaN


In [475]:
#nom des colonnes numériques
cols_num = (
    df_dispo_alimentaire_raw
    .select_dtypes(include='number')
    .columns
)
cols_num

Index(['Aliments pour animaux', 'Autres Utilisations',
       'Disponibilité alimentaire (Kcal/personne/jour)',
       'Disponibilité alimentaire en quantité (kg/personne/an)',
       'Disponibilité de matière grasse en quantité (g/personne/jour)',
       'Disponibilité de protéines en quantité (g/personne/jour)',
       'Disponibilité intérieure', 'Exportations - Quantité',
       'Importations - Quantité', 'Nourriture', 'Pertes', 'Production',
       'Semences', 'Traitement', 'Variation de stock'],
      dtype='object')

In [476]:
#nom des colonnes catégorielles
cols_cat = (
    df_dispo_alimentaire_raw
    .select_dtypes(include='object')
    .columns
)
cols_cat

Index(['Zone', 'Produit', 'Origine'], dtype='object')

#### 2.2.2 Compréhension des variables

In [477]:
#nombre de pays
df_dispo_alimentaire_raw['Zone'].nunique()

174

In [478]:
df_dispo_alimentaire_raw['Zone'].unique()

array(['Afghanistan', 'Afrique du Sud', 'Albanie', 'Algérie', 'Allemagne',
       'Angola', 'Antigua-et-Barbuda', 'Arabie saoudite', 'Argentine',
       'Arménie', 'Australie', 'Autriche', 'Azerbaïdjan', 'Bahamas',
       'Bangladesh', 'Barbade', 'Belgique', 'Belize', 'Bermudes',
       'Bolivie (État plurinational de)', 'Bosnie-Herzégovine',
       'Botswana', 'Brunéi Darussalam', 'Brésil', 'Bulgarie',
       'Burkina Faso', 'Bélarus', 'Bénin', 'Cabo Verde', 'Cambodge',
       'Cameroun', 'Canada', 'Chili', 'Chine - RAS de Hong-Kong',
       'Chine - RAS de Macao', 'Chine, Taiwan Province de',
       'Chine, continentale', 'Chypre', 'Colombie', 'Congo', 'Costa Rica',
       'Croatie', 'Cuba', "Côte d'Ivoire", 'Danemark', 'Djibouti',
       'Dominique', 'El Salvador', 'Espagne', 'Estonie', 'Eswatini',
       'Fidji', 'Finlande', 'France', 'Fédération de Russie', 'Gabon',
       'Gambie', 'Ghana', 'Grenade', 'Grèce', 'Guatemala', 'Guinée',
       'Guinée-Bissau', 'Guyana', 'Géorgie', 'H

In [479]:
#nombre de produits disponibles
df_dispo_alimentaire_raw['Produit'].nunique()

98

In [480]:
#type de produits/origine animale ou végétale
df_dispo_alimentaire_raw['Origine'].value_counts()

Origine
vegetale    11896
animale      3709
Name: count, dtype: int64

In [481]:
#distribution globale des variables numériques
df_dispo_alimentaire_raw.describe(include='number')

,Aliments pour animaux,Autres Utilisations,Disponibilité alimentaire (Kcal/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Pertes,Production,Semences,Traitement,Variation de stock
count,2720,5496,14241,14015,11794,11561,15382,12226,14852,14015,4278,9180,2091,2292,6776
mean,480,157,35,9,1,1,640,111,87,348,106,1090,74,962,-15
std,4240,5077,107,25,4,4,9067,1053,717,4476,1113,12067,528,10382,550
min,0,0,-21,-2,-0,-0,-3430,-41,-201,-246,0,0,0,-19,-39863
25%,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0
50%,4,0,4,1,0,0,7,0,2,5,4,22,2,6,0
75%,74,4,21,5,1,1,77,9,18,52,26,191,17,69,0
max,150000,347309,1711,431,61,55,739267,42797,63381,426850,55047,739267,17060,326711,5284


#### 2.2.3 Audit de qualité

In [482]:
# pourcentage de valeurs manquantes
(df_dispo_alimentaire_raw.isna().sum()/nb_lignes*100).round(2)

Zone                                                             0
Produit                                                          0
Origine                                                          0
Aliments pour animaux                                           83
Autres Utilisations                                             65
Disponibilité alimentaire (Kcal/personne/jour)                   9
Disponibilité alimentaire en quantité (kg/personne/an)          10
Disponibilité de matière grasse en quantité (g/personne/jour)   24
Disponibilité de protéines en quantité (g/personne/jour)        26
Disponibilité intérieure                                         1
Exportations - Quantité                                         22
Importations - Quantité                                          5
Nourriture                                                      10
Pertes                                                          73
Production                                                    

In [483]:
#vérification doublons métier
df_dispo_alimentaire_raw[['Zone', 'Produit']].duplicated().sum()

np.int64(0)

Conclusion :
* Chaque ligne représente un pays et un produit unique
* Types adéquats
* Plusieurs colonnes présentent un pourcentage élevé de valeur manquante :
    - Aliments pour animaux ~82.6%
    - Autres utilisations ~64.8%
    - Pertes ~72.6%
    - Semences ~86.6%
    - Traitement ~85.3%
    - Variation de stock ~56.6%

Décider une règle métier


#### 2.2.4 Nettoyage et standardisation 

In [484]:
#standardisation des zones
df_dispo_alimentaire_raw['Zone'] = (
    df_dispo_alimentaire_raw['Zone']
    .str.strip()
    .str.title()
)

In [485]:
#produits disponibles
df_dispo_alimentaire_raw['Produit'].unique()

array(['Abats Comestible', 'Agrumes, Autres', 'Aliments pour enfants',
       'Ananas', 'Bananes', 'Beurre, Ghee', 'Bière', 'Blé',
       'Boissons Alcooliques', 'Café', 'Coco (Incl Coprah)', 'Crème',
       'Céréales, Autres', 'Dattes', 'Edulcorants Autres',
       'Feve de Cacao', 'Fruits, Autres', 'Graines de coton',
       'Graines de tournesol', 'Graisses Animales Crue',
       'Huil Plantes Oleif Autr', 'Huile Graines de Coton',
       "Huile d'Arachide", "Huile d'Olive", 'Huile de Colza&Moutarde',
       'Huile de Palme', 'Huile de Soja', 'Huile de Sésame',
       'Huile de Tournesol', 'Lait - Excl Beurre', 'Légumes, Autres',
       'Légumineuses Autres', 'Maïs', 'Miel', 'Millet', 'Miscellanees',
       'Noix', 'Oeufs', 'Olives', 'Oranges, Mandarines', 'Orge',
       'Plantes Oleiferes, Autre', 'Poissons Eau Douce', 'Poivre',
       'Pommes', 'Pommes de Terre', 'Raisin', 'Riz (Eq Blanchi)',
       'Sucre Eq Brut', 'Sucre, betterave', 'Sucre, canne', 'Sésame',
       'Thé', 'Toma

In [486]:
#standardisation produits
df_dispo_alimentaire_raw['Produit'] = (
    df_dispo_alimentaire_raw['Produit']
    .str.strip()
    .str.capitalize()
)

In [487]:
#correction des liballés produits

correction_produits_dispo = {
    'Agrumes, autres':'Agrumes et autres',
    'Beurre, ghee':'Beurre et ghee',
    'Coco (incl coprah)':'Coco (inclut coprah)',
    'Céréales, autres':'Céréales et autres',
    'Fruits, autres':'Fruits et autres',
    'Huil plantes oleif autr':'Huile de plantes oléifères et autres',
    'Huile de colza&moutarde':'Huile de colza et moutarde',
    'Lait - excl beurre':'Lait (exclut beurre)',
    'Légumes, autres':'Légumes et autres',
    'Légumineuses autres':'Légumineuses et autres',
    'Oranges, mandarines':'Oranges et mandarines',
    'Plantes oleiferes, autre':'Plantes oleiferes et autres',
    'Poissons eau douce':'Poissons d\'eau douce',
    'Riz (eq blanchi)':'Riz (équivalent blanchi)',
    'Sucre eq brut':'Sucre (équivalent brut)',
    'Sucre, betterave':'Sucre de betterave',
    'Sucre, canne':'Sucre de canne',
    "Viande d'ovins/caprins":"Viande d'ovins et caprins",
    'Viande, autre':'Viande et autres',
    'Épices, autres':'Épices et autres',
    'Animaux aquatiques autre':'Animaux aquatiques et autres',
    'Citrons & limes':'Citrons et limes',
    'Graines colza/moutarde':'Graines colza et moutarde',
    'Huiles de foie de poisso':'Huiles de foie de poisson',
    'Mollusques, autres':'Mollusques et autres',
    'Poissons marins, autres':'Poissons marins et autres',
    'Racines nda':'Racines (non définies)',
    'Viande de suides':'Viande de suidés',
    'Viande de anim aquatiq':'Viande d\'animale aquatique'
}
df_dispo_alimentaire_raw['Produit'] = (
    df_dispo_alimentaire_raw['Produit']
    .replace(correction_produits_dispo)
)

In [488]:
# nouvelle variable - catégorie

#définir les catégories

categories_fao = {
    'Céréales': [
        'Blé',
        'Céréales et autres',
        'Maïs',
        'Millet',
        'Orge',
        'Riz (équivalent blanchi)',
        'Avoine',
        'Seigle',
        'Sorgho'
        ],
    'Légumineuses': [
        'Légumineuses et autres',
        'Haricots'
        ],
    'Racines et tubercules': [
        'Pommes de terre',
        'Ignames',
        'Manioc',
        'Patates douces',
        'Racines (non définies)'
        ],
    'Fruits': [
        'Agrumes et autres', 
        'Ananas', 
        'Bananes',
        'Dattes',
        'Fruits et autres',
        'Oranges et mandarines',
        'Pommes',
        'Raisin',
        'Bananes plantains',
        'Citrons et limes',
        'Pamplemousse'
        ],
    'Légumes': [
        'Légumes et autres',
        'Tomates',
        'Oignons',
        'Pois'
        ],
    'Oléagineux': [
        'Coco (inclut coprah)',
        'Graines de coton',
        'Graines de tournesol',
        'Olives',
        'Plantes oleiferes et autres',
        'Sésame',
        'Arachides decortiquees',
        'Graines colza et moutarde',
        'Palmistes',
        'Soja'
        ],
    'Boissons stimulantes': [
        'Café',
        'Feve de cacao',
        'Thé'
        ],
    'Epices': [
        'Poivre',
        'Épices et autres',
        'Girofles',
        'Piments'
        ],
    'Sucres et édulcorants': [
        'Edulcorants autres',
        'Miel',
        'Sucre (équivalent brut)', 
        'Sucre de betterave', 
        'Sucre de canne',
        'Sucre non centrifugé'
        ],
    'Lait et fromage' : [
        'Crème',
        'Lait (exclut beurre)'
        ],
    'Oeufs': [
        'Oeufs'
        ],
    'Huiles et graisses': [
        'Beurre et ghee',
        'Graisses animales crue',
        'Huile de plantes oléifères et autres', 
        'Huile graines de coton',
        "Huile d'arachide", 
        "Huile d'olive", 
        'Huile de colza et moutarde',
        'Huile de palme', 
        'Huile de soja', 
        'Huile de sésame',
        'Huile de tournesol',
        'Huile de coco',
        'Huile de germe de maïs', 
        'Huile de palmistes',
        'Huiles de foie de poisson', 
        'Huiles de poissons',
        'Huile de son de riz'
        ],
    'Fruit à coques':[
        'Noix'
        ],
    'Viandes':[
        'Abats comestible', 
        "Viande d'ovins et caprins",
        'Viande de bovins', 
        'Viande de volailles', 
        'Viande et autres',
        'Viande de suidés',
        "Viande d'animale aquatique"
        ],
    'Poisson et fruits de mer':[
        "Poissons d'eau douce",
        'Animaux aquatiques et autres',
        'Cephalopodes',
        'Crustacés',
        'Mollusques et autres',
        'Perciform',
        'Plantes aquatiques',
        'Poissons marins et autres',
        'Poissons pelagiques'
        ],
    'Boissons alcoolisees': [
        'Bière',
        'Boissons alcooliques',
        'Vin',
        'Alcool, non comestible',
        'Boissons fermentés'
        ],
    'Produits divers':[
        'Miscellanees',
        'Aliments pour enfants'
    ]
}

#Créer un mapping automatique
mapping = {}

for categorie, produits in categories_fao.items():
    for produit in produits:
        mapping[produit] = categorie

#créer la colonne
df_dispo_alimentaire_raw['Categorie'] = (
    df_dispo_alimentaire_raw['Produit']
    .map(mapping)
)
len(mapping)

98

In [489]:
#standardisation origine
df_dispo_alimentaire_raw['Origine'] = (
    df_dispo_alimentaire_raw['Origine']
    .str.strip()
    .str.capitalize()
)

In [490]:
#choix métier : remplacer les valeurs manquantes par 0 (zero)
df_dispo_alimentaire_raw = (
    df_dispo_alimentaire_raw
    .fillna(0)
)

In [491]:
#validation 
df_dispo_alimentaire_raw.isna().sum()

Zone                                                             0
Produit                                                          0
Origine                                                          0
Aliments pour animaux                                            0
Autres Utilisations                                              0
Disponibilité alimentaire (Kcal/personne/jour)                   0
Disponibilité alimentaire en quantité (kg/personne/an)           0
Disponibilité de matière grasse en quantité (g/personne/jour)    0
Disponibilité de protéines en quantité (g/personne/jour)         0
Disponibilité intérieure                                         0
Exportations - Quantité                                          0
Importations - Quantité                                          0
Nourriture                                                       0
Pertes                                                           0
Production                                                    

In [492]:
#transformation des unités 
#selection des colonnes à convertir, except colonnes 'Disponibilité'
cols_utilisations = (
    cols_num
    .drop(
        df_dispo_alimentaire_raw
        .filter(like='Disponibilité')
        .columns
        )
)
#converstion milliers de tonnes en kg
df_dispo_alimentaire_raw[cols_utilisations] *= 1_000_000
df_dispo_alimentaire_raw['Disponibilité intérieure'] *= 1_000_000

#### 2.2.5 Validation des transformations

In [493]:
#aperçu après nettoyage
df_dispo_alimentaire_raw.head()

,Zone,Produit,Origine,Aliments pour animaux,Autres Utilisations,Disponibilité alimentaire (Kcal/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Pertes,Production,Semences,Traitement,Variation de stock,Categorie
0,Afghanistan,Abats comestible,Animale,0,0,5,2,0,1,53000000,0,0,53000000,0,53000000,0,0,0,Viandes
1,Afghanistan,Agrumes et autres,Vegetale,0,0,1,1,0,0,41000000,2000000,40000000,39000000,2000000,3000000,0,0,0,Fruits
2,Afghanistan,Aliments pour enfants,Vegetale,0,0,1,0,0,0,2000000,0,2000000,2000000,0,0,0,0,0,Produits divers
3,Afghanistan,Ananas,Vegetale,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Fruits
4,Afghanistan,Bananes,Vegetale,0,0,4,3,0,0,82000000,0,82000000,82000000,0,0,0,0,0,Fruits


In [494]:
df_dispo_alimentaire_raw.to_csv('../data/data_cleaned/dispo_alimentaire_cleaned.csv', index=False)
df_dispo_alimentaire = pd.read_csv('../data/data_cleaned/dispo_alimentaire_cleaned.csv')

In [495]:
#dimensions du dataset après les transformations
print(f'Après les transformations, le dataframe dispo_alimentaire comporte :\n{df_dispo_alimentaire.shape[0]} enregistrements et {df_dispo_alimentaire.shape[1]} colonnes')

Après les transformations, le dataframe dispo_alimentaire comporte :
15605 enregistrements et 19 colonnes


#### 2.2.6 Transformations réalisées

* Standardisation des variables texte (suppression des espaces inutiles, uniformisation de l'écriture et correction des erreurs orthographiques).
* Conversion des variables d'utilisation et de la disponibilité intérieure en kilogrammes (1 millier de tonnes = 1 000 000 kg)
* Remplacement des valeurs manquantes par zéro.


___________

### 2.3 Fichier population

**Description du dataset**

Ce dataset présente la population totale de 174 pays entre 2013 et 2018. 

Chaque ligne représente la population totale d'un pays pour une année donnée.

Après nettoyage et transformations, le dataframe population conserve 1416 enregistrements et 3 colonnes.

#### 2.3.1 Compréhension générale du dataset

In [496]:
#dimensions du dataset
print(f'Le dataframe population comporte :\n{df_population_raw.shape[0]} enregistrements et {df_population_raw.shape[1]} colonnes')

Le dataframe population comporte :
1416 enregistrements et 3 colonnes


In [497]:
#structure des colonnes et types
df_population_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1416 entries, 0 to 1415
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Zone    1416 non-null   object 
 1   Année   1416 non-null   int64  
 2   Valeur  1416 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 33.3+ KB


In [498]:
#aperçu des données
df_population_raw.head()

,Zone,Année,Valeur
0,Afghanistan,2013,32270
1,Afghanistan,2014,33371
2,Afghanistan,2015,34414
3,Afghanistan,2016,35383
4,Afghanistan,2017,36296


In [499]:
#nom des colonnes
df_population_raw.columns

Index(['Zone', 'Année', 'Valeur'], dtype='object')

#### 2.3.2 Compréhension des variables

In [500]:
#nombre de pays
df_population_raw['Zone'].nunique()

236

In [501]:
#années disponibles
df_population_raw['Année'].unique()

array([2013, 2014, 2015, 2016, 2017, 2018])

#### 2.3.3 Audit de qualité

In [502]:
#valeurs manquantes
df_population_raw.isna().sum()

Zone      0
Année     0
Valeur    0
dtype: int64

In [503]:
#vérification doublons métier
df_population_raw[['Zone','Année']].duplicated().sum()

np.int64(0)

Conclusion :
* Chaque ligne du dataset représente la population total d'un pays pour une année donnée.
* Pas de valeur manquantes.
* Types adéquats.

#### 2.3.4 Nettoyage et standardisation

In [504]:
#nettoyer et standardiser variable Zone
df_population_raw['Zone'] = (
    df_population_raw['Zone']
    .str
    .strip()
    .str
    .title()
)

In [505]:
#changer les valeurs d'une colonne
df_population_raw['Population'] = (
    df_population_raw['Valeur']*1000
)

In [506]:
#supprimer une colonne
df_population_raw.drop(
    columns='Valeur', 
    inplace=True
    )

#### 2.3.5 Validation des transformations

In [507]:
#aperçu après nettoyage
df_population_raw.head()

,Zone,Année,Population
0,Afghanistan,2013,32269589
1,Afghanistan,2014,33370794
2,Afghanistan,2015,34413603
3,Afghanistan,2016,35383032
4,Afghanistan,2017,36296113


In [508]:
df_population_raw.to_csv('../data/data_cleaned/population_cleaned.csv', index=False)
df_population = pd.read_csv('../data/data_cleaned/population_cleaned.csv')

In [509]:
#dimensions du dataset après transformation
print(f'Après les transformations, le dataframe population comporte :\n{df_population.shape[0]} enregistrements et {df_population.shape[1]} colonnes')

Après les transformations, le dataframe population comporte :
1416 enregistrements et 3 colonnes


#### 2.3.6 Transformations réalisées

* Standardisation des variables texte (suppression des espaces inutiles, uniformisation de l'écriture et correction des erreurs orthographiques).
* Renommage de la variable Valeur en Population.
* Harmonisation des unités de la variable Population (multiplication par 1000).

______

### 2.4 Fichier sous nutrition

**Description du dataset**

Ce dataset présente nombre moyen de personnes en sous-alimentation sur trois années, en millions d’habitants. 

Chaque intervalle peut être représenté par son année centrale (par exemple, l’intervalle 2012-2014 est associé à l’année de référence 2013).

Chaque ligne correspond à un couple unique pays, période.

Initialement, le dataframe sous_nutrition comportait 1218 enregistrements et 3 colonnes. Après les transformations, il contient 1218 enregistrements et 6 colonnes.

#### 2.4.1 Compréhension générale du dataset

In [510]:
#dimension du dataset
print(f'Le dataframe sous_nutrition présente :\n{df_sous_nutrition_raw.shape[0]} enregistrements et {df_sous_nutrition_raw.shape[1]} colonnes')

Le dataframe sous_nutrition présente :
1218 enregistrements et 3 colonnes


In [511]:
#structure des colonnes et types
df_sous_nutrition_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Zone    1218 non-null   object
 1   Année   1218 non-null   object
 2   Valeur  624 non-null    object
dtypes: object(3)
memory usage: 28.7+ KB


In [512]:
#aperçu des données
df_sous_nutrition_raw.head()

,Zone,Année,Valeur
0,Afghanistan,2012-2014,8.6
1,Afghanistan,2013-2015,8.8
2,Afghanistan,2014-2016,8.9
3,Afghanistan,2015-2017,9.7
4,Afghanistan,2016-2018,10.5


In [513]:
#nom des colonnes
df_sous_nutrition_raw.columns

Index(['Zone', 'Année', 'Valeur'], dtype='object')

#### 2.4.2 Compréhension des variables

In [514]:
#nombre de pays/zones
df_sous_nutrition_raw['Zone'].nunique()

203

In [515]:
df_sous_nutrition_raw['Année'].unique().tolist()

['2012-2014', '2013-2015', '2014-2016', '2015-2017', '2016-2018', '2017-2019']

#### 2.4.3 Audit de qualité

In [516]:
#proportion de valeurs manquantes
(df_sous_nutrition_raw.isna().sum()/df_sous_nutrition_raw.shape[0]*100).round(2)

Zone      0
Année     0
Valeur   49
dtype: float64

In [517]:
#vérification doublons métier
df_sous_nutrition_raw[['Zone','Année']].duplicated().sum()

np.int64(0)

Conclusion :
* Chaque ligne représente le total de personnes d'un pays en sous-nutrition dans un intervalle de 3 ans.
* Valeurs manquantes variable 'Valeur' (~48.7%).
* Changer type de la variable 'Valeur' (to_numeric).

#### 2.4.4 Nettoyage et standardisation

In [518]:
#nettoyer et standardiser variable Zone
df_sous_nutrition_raw['Zone'] = (
    df_sous_nutrition_raw['Zone']
    .str
    .strip()
    .str
    .title()
)

In [519]:
#changer le type de la variable Valeur
df_sous_nutrition_raw['Valeur'] = pd.to_numeric(
    df_sous_nutrition_raw['Valeur'], 
    errors='coerce'
)

In [520]:
#vérifier si bon type
df_sous_nutrition_raw['Valeur'].dtype

dtype('float64')

In [521]:
#remaplacer les valeurs manquantes NaN par 0 (zero)
df_sous_nutrition_raw['Valeur'] = (
    df_sous_nutrition_raw['Valeur']
    .fillna(0)
)

In [522]:
#vérifier si tous les valeurs manquantes ont été remplacés
df_sous_nutrition_raw['Valeur'].isna().sum()

np.int64(0)

In [523]:
#renomer une colonne
df_sous_nutrition_raw.rename(
    columns={'Valeur':'sous_nutrition'}, 
    inplace=True
    )

In [524]:
#changer les valeurs d'une colonne
df_sous_nutrition_raw['sous_nutrition'] *= 1_000_000

In [525]:
# renomer colonne année
df_sous_nutrition_raw.rename(
    columns={'Année':'Année_intervalle'}, 
    inplace=True
    )

In [526]:
#transformer la colonne année en deux variables
df_sous_nutrition_raw[['annee_debut','annee_fin']] = (
    df_sous_nutrition_raw['Année_intervalle']
    .str
    .split('-', expand=True)
    .astype(int)
)


In [527]:
#créer la colonne année de référence 
df_sous_nutrition_raw['annee_reference'] = (
    df_sous_nutrition_raw['annee_debut'] + df_sous_nutrition_raw['annee_fin']
    )//2

#### 2.4.5 Validation des transformations

In [528]:
#aperçu après nettoyage
df_sous_nutrition_raw.head()

,Zone,Année_intervalle,sous_nutrition,annee_debut,annee_fin,annee_reference
0,Afghanistan,2012-2014,8600000,2012,2014,2013
1,Afghanistan,2013-2015,8800000,2013,2015,2014
2,Afghanistan,2014-2016,8900000,2014,2016,2015
3,Afghanistan,2015-2017,9700000,2015,2017,2016
4,Afghanistan,2016-2018,10500000,2016,2018,2017


In [529]:
df_sous_nutrition_raw.to_csv('../data/data_cleaned/sous_nutrition_cleaned.csv', index=False)
df_sous_nutrition = pd.read_csv('../data/data_cleaned/sous_nutrition_cleaned.csv')

In [530]:
#dimension du dataset après les transformations
print(f'Après les transformations, le dataframe sous_nutrition présente :\n{df_sous_nutrition.shape[0]} enregistrements et {df_sous_nutrition.shape[1]} colonnes')

Après les transformations, le dataframe sous_nutrition présente :
1218 enregistrements et 6 colonnes


#### 2.4.6 Transformations réalisées

* Standardisation des variables texte (suppression des espaces inutiles, uniformisation de l'écriture et correction des erreurs orthographiques).
* Conversion de la variable Valeur en type numérique.
* Remplacement des valeurs manquantes par zéro.
* Renommage de la variable Valeur en sous_nutrition.
* Multiplication de la colonne sous_nutrition par 1 000 000 (millions d'habitants).
* Création des variables annee_reference, annee_debut et annee_fin.
* Renommage de la variable Année en annee_intervalle.

_______________